# Phase 10: Advanced Transformer Intent Model (DistilRoBERTa)
### Google Colab GPU Training & Golden Evaluation Pipeline

This notebook fine-tunes a pretrained **DistilRoBERTa** (`distilroberta-base`) model on the isolated AppleSupport intent training dataset and benchmarks performance against the human-verified **Golden Evaluation Set**.

**Key Constraints:**
- **Hardware Requirement:** Execution **MUST** run on a CUDA GPU (e.g. Tesla T4, V100, A100). CPU training is prohibited.
- **Strict Training Isolation:** Asserts $S_{\text{train}} \cap S_{\text{golden}} = \emptyset$.
- **Zero Data Leakage:** Only initial customer message text is used. No agent replies or future conversation turns.
- **Reproducibility:** Deterministic training with fixed seed (`42`).

In [ ]:
# Cell 1: Pre-Flight GPU Hardware Validation
import sys
import subprocess

print("=== Checking GPU Availability via nvidia-smi ===")
try:
    nvidia_smi = subprocess.check_output(["nvidia-smi"]).decode("utf-8")
    print(nvidia_smi)
except Exception as e:
    print(f"nvidia-smi failed: {e}")

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "CRITICAL ERROR: CUDA GPU is not available in this runtime!\n"
        "Please switch to a GPU runtime in Colab: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"\nSUCCESS: Connected to GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
print(f"PyTorch Version: {torch.__version__}, CUDA Version: {torch.version.cuda}")

In [ ]:
# Cell 2: Install Required Deep Learning Dependencies
!pip install -q --upgrade transformers datasets accelerate evaluate scikit-learn pandas scipy

In [ ]:
# Cell 3: Environment Setup, Repository Synchronization & Dataset Resolution
import os
import sys
import gzip
import base64
import shutil
from pathlib import Path

REPO_NAME = "ai-customer-agent"
CURRENT_DIR = Path.cwd()

# 1. Ensure repository code directory structure is active
if not (CURRENT_DIR / "src").exists():
    if not Path(REPO_NAME).exists():
        print(f"Cloning {REPO_NAME} from GitHub...")
        !git clone https://github.com/mohanraj9342/ai-customer-agent.git
    os.chdir(REPO_NAME)

print(f"Active working directory: {os.getcwd()}")
Path("src/classification").mkdir(parents=True, exist_ok=True)
Path("models/distilroberta_intent_classifier").mkdir(parents=True, exist_ok=True)
TARGET_DATA_DIR = Path("data/processed/apple_support")
TARGET_DATA_DIR.mkdir(parents=True, exist_ok=True)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# 2. Automatically synchronize/extract Python helper modules if needed
train_py = Path("src/classification/train_distilroberta.py")
eval_py = Path("src/classification/eval_transformer.py")

if not train_py.exists():
    print("Unpacking train_distilroberta.py...")
    train_py.write_bytes(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/61c/XLcRnL/f59iDi7FgAoLiZJ8cTaFq6L5ITGhJBZJ2bkwLBwIzC5xwgIIgKW4dvRnqvIqea08Sbp7vgHsSirbdSctMD09PT3TPb/uacjzvFnXZs+yMu26YllkaV/U1bO+TYsqyYuuL8q2vuNtn0bNdhZ/7X+zY84bdt2mVbes2zVv2WlR8fn1piqqFbsoGl7CM/OPaYTL+qeTy+s0YEDLDpum5Febpqnbnp1VPa96duRIF81m/8q37PUmBf49591iNmdXfVtkPXt98YGdVMAn42vouWCHXQfSd6yHV/dRtsnTqOiS9CEtyvSu5H7A7jiQc0ZTRunu+KqoumjG2HmdpSU7Apa6sehYRyOVW9a09X1xV/Q8Zw3MsIY/SD7os9oUOU0R+GjZrhWTs64uiXLBfk7LIk973rGrhAZh//c//wu/V3WZ84rF8PjfSsIlqrAnFSLXs6rZAFP+2LOTx6zcdMVD0W8X7H0FssFAfQGCZJuur1H/ICxMdw3Sw2iomo75f+uh898ClgLzTcdz5EqscpDn+4rzvEta/lDwT9+zN3zT4mJl7KiuliXMp1uwtzXMEnVCooMetG7qih0csKyuspb3oNz0sa7q9RZ+tCvek1ZoUee/8GJ137Of0jKtMpjZgok3wOyorYHgpOrbutmyEh5YmucthxWFVeAPHMSmfcuK9R3158j3sO2LZQr6vuBtByJzeL9gV7wFfRS/QlelRxjiE43VhbA9PvIKWtsQhV4Wq5ClVW7Wfc37FNYpxQFei7W54qD4h7TcyKU8qtewIMA/A1F5Pv9Ut2WOHUElMALsjnkhtnPLcXPDuw70yA0Jjpiu74rVpt50LC/SVVWjykFdHtjpbNnWa5Yky02/aXmSwLTJSNKqqnsSopvN1Lt21aRtx9Xz37u6Ur/LegU7fKUe60796radGAN3ZF+suRpBPYOe4M9f64oLuibt78viTpFdwKNo6LcNWYt4f1htQ3YMGyNk57AgIajuPze4LCG73oC1a6mrzRpWOu1Y1ahXDSgFXsD/mlzw7j6WPG2rSKpNDeKDwTKWZtmmTbNt0mVgMiG9c31bIrQvm2CtwXDg7ToFbo/ibdPyrOgELXiAMlkSt6QTXimcBQNJ0AySjpc8wxGURMKJwo7ok64pi14uILjbyBUputsUZS5tPuF6TwHL3p3e0eHVSXJ8dnp6dvTh/PqvyfXZyeWVELqs0x0sRDuYS7HcJmpHJ4VyQmY6Y8HEhk2QN/gQKQoYcCIbqnTNYdPhjoL2WG2tCEjO6Z2fEE2SBLPZ8cnpIUidXF8enr1Ljq5+hg4eGtUzcE4ZmnX+LEXvrxTtPqkx9Qwy2BrCeUZZ9+Bp/q/fnx+f/I4BpBJBcy7f9x+uLz5cg/ovkS8tevfMOSMVB6VF3preJz8fnif/cvX+3TcK5Q5grSw4wk0JrhRN2zPaffv++OQ8eXf49oQGsnvP79KOG4muTk6OgebVC9P38N+S85N3r6/fwPuDFz/qhp8Or4/eJFdn/45MX5oO5yeHl+/O3r1OLg+vqYnPfzAzvnh/9OYK3+pXvxxevv1wgdRn76HheXRgmk7OXr8B7Z4cHf6Vmp4fzGaznC9pv8FhDSa2BtXAyndwas//InzHDfqVGzh3Qjjz+lvhZ27gZ4iH0e3tgjY/uM9LDl6zYrBp6qrAc51YvihycrtF/oKemRqCsEh/z/EY06cXrStsN/TGyNa2ApC5g/XiuT+0Dz8IhIWqAWP2GzYsWEGjFCHDR+DGOHhARBHctxkEn8VoSkboXyxEn2/p3woNKDFCzVDqWXqIVbNx8BFo2ugYXLlR6c/UAbSUwoHDjj4cHxL+Ajyg+ws1WjhQWS+AOORymRZ4ml9uKjxZTtoW57MUvIDPptKcoh1wzKCwSMlFf/ftVghKqpMeGUEgveSPGW8AX9J7MSwcMfDW9GlRNEcyX7fRQBfba+SHQsABDOoHbFWWgCpgIXDnwEnU4nHPq4eirSsEXRHzBjzgBIFReNXBkc629YbAWLupxPQEoyPw1Hek2lZIExkuAJrRd4PkQqGgPBRmJ9z9hvkdXZ5dnx0dni+cpUXumt+f9uBqsd1AHadp2fHxzAeQuQVMULQCmqIVW4EBWwOKBRSsYSacsamQpwI0Vq9KqaPRIF+N4G2NCkXmAHwznmQ1KAgszpqn3eIHNjGZoUOLvsBq9J879FnapHcFQIPtzl6GRPXtAeyBK+TrugVjvYOeLUiS+9P9YYqg5L4AN/Q8iOy+7BnzD56/ePX06csgZC/ktNH6Cwig0MlolXjI1Syut4BYZiPRFbXbKoFW+3FMhYowRPhk0WQCRVvzBtKl99tIHTfPbz9HE68Pbj97Fr+BtoDZ4I1DCxpMHjBsgCN1IVckUW+SRNB+lngL8U2EuvI93IvSkhh5xYLnC/YEYiz4/+ufmBgtFIZkrfqTLrCEVf/ZmhmJq1foZkpXt4Ht6hWpdPCAayEo4AmiD0A3nS8dJcJUQDoJwvkFWgf7L8LyQjIJh/LlAiB4dAx9T1u9ZgBHEjJhWCRAib041J+Lxo6jFgqyHxt0AOI057fD0x2B7T/b4dhrIogO2ja1DqVzwKrG2g1GDMHJUl6AGeQLoWDZox+CQNqNd0UwRrBdMHsGM43M2YIqQ3XBzFBTvqvEwHHFmjjij+DWurETPi1K/q7uT9GOhSdeetfjKTCEtMhyiYQs7RfsN8P9sydN2NmZqA7kMqEROjlwl3qhJaR0UEtY1k8wO1iQlkNkAVPzbSKi+o4dRCq7AQHUPYc/CFtpFbPsnmcfF3uyG/uDE18IEppdGIznKCWwXLwa/UHbIoT2NavhuUwb9qmAlZNCIMz39HxeRCoDAtgDEHZdPsB5swagXcwhJuIrdJyZzICA4/0EJu7uHK2/DI52nKOYwY38y9P6lxjRu2V/AqBu8/Buo6xutv54qsZXndLOxbMQ9u6TfJiuQcn+mQkR9IisqeuS3QPQgQ4Q2tZtDljW+J+SK32Dfaon4hGMiOzXWnsvAaQ5KNrBvaGNYKdA/YTm6OeN/jGhPYAdReWrMaKPfIuA29Gg6U5koHGL9xRLEElzHLLAnNmAg3gVgTMpq9T3vCBKu37bcB+8lNbNq0hliJhMVuUyeyWTUASVKcslXmCKRqcvxFnaOcPKyUQYEsrDFqYedSKUzfmjnD6EsukaXC32dlZPtsqoRra681ayxehr0YY6X3cIWY6zjKGFfP/LF6IPziNLCsLBgPxX3HSxPF9Gx4I9OYQtPnYM2UGg6aQEN9iAajezAfxihH+KDCd8g9R5vk/pT3IT2YVGIXrpfiAnBx4FXYk5DnDToJMRyR25S6g1xF9AgahukAQyJqyWwZiWoCp+5bE+VkPrpKjyep1Q/jbGk9U0dUK4bTzaG8o+R2o5FhBAyL5ADQj/LHUb4ht7iuI1qkfuICIP9CPQBlJfEnsMdRFO+IFQLYMEKBPXDl/CKBa00HklB7d8oZNJFole4E5hv4AM7Y4OJgskOojkn4jnsYOhNJkYQcmbOrvvhoBIZEkExV3aAwDFHTCkMgkYme3DxCOelBjsG+w1mZeRcqaPCSzWCjXh8jZZn2+AbZTT1YldkxwwQOxUpdkHwRxsJ4TLmw6P6jcbShiepoDLCPEAGqPTmWxw/kndBaBf1PCLYIe85SFeMjIhCDyMYqazGtZ5T4BRggGTXkNc4MJfSm/uyrH6g91mH4kXAnZjFj0lWClEMjBl0ndM24vwl8YrT0P6scnE7qPxHXp6sf5lORbYCZavkZMaJVPUnQFJoJqlhzEEVhZokLTH/w43fU3XSad1q64H3Gu/0KG91hc2+rXcP4MXsLsO29VG5BWsmYGnHU5MvfTxD/vg1oPtwdeKRAJq4xZU1C7bYdEc+SNUDoTpXCY3fLunTBYsdfeEXIRPfy4s08N7lRuyv50JO8s3a2HcjA9xVXDGDUubNMdpxp5xI4PAtW83lVio2E0OuM4nNj/DYcrlO7xbBOPAveV4BrmXBJwElCvOqli9lyqkmyJ9Nt3IiYRMHYYyNob+4mje0RsaJ/uO116uoQ6vhHGbU1OZRBRhhGHLnsAKIJaTjwQ43SUOxWqA/aE2bclNXwxJv9zTgBiyr/E0zsRdsZhIJ21vcGsm9rNMaWi4FDgbPXTxY2COR7nn99v3yBD09rCG0O+qzVrED13sjGoolNOMtfc0MYykjrWLdeOYPwNWp4tz90yyb6xH99Pn8GDwur7Flgx8+bcFgNGskwQv6pMEPE65DNnTtF0BQKBj1fH0ACArwPUqJXVNT4BN3tUVJhzxL+j99OMnzYDcAL5fOIbYbRoQJoj0wDSk6Ru41CBVNDxypgSbObNSeSk8tOXM5PahmRVYuNAtBk4qlM4pEQgM2u8wWo1F+pimA2TubMQGABrBkgIIYbGd585EMgVSksR/+lR0canALAoikuSSIb0dMCyWU9qR+fGx3gV3oFxmJptcVRHtIbmFcAf5glM85h31tS8kkQnoIJgcgWCKGEjRY0rAnx+EQmJSQCSqHCJjRoEEHJp8wB9adszJ6AviDnj2bh0yeeb42BQqygAV6C44jUAMzbGndpK86/cRdKGLyM0eN7uIQLB12InZh2aX6N6aBB9kcAu7H44mX3VKH4sutsLQNMuAzi0vEF4HazuQjaFdp1lbJ8sD6LA8mCIF9nDMpSuO5ynQghvF2BqijgeqO4ifD+Nfnn8lP0W+j6Vckd/cuxI1N0+GEz68CNxj3FMz0zTqxZDQklrTWu8s8s/K6f5jZCqkNGJTMZmTbDUxWuC0R+uP8M5HQAxdBQxhlHNN6o/WOUo5yaaGCKdDJmL/ihGeMc9q9WZ2EkT6Om2+4tF3rFTlRASFTouYIBZjNnS0wGUET82JZ2YYw9b2BwJb2qMtTfE/X21jj0JMC5h1sC/2NDtRZOw8GSI4MNRlksAuJkqNzc9JepJukpw9ZS/cw1zwFjFyLP4KB0YAbLN0G09VDFikabveNCJ7Ek/VHRjSZXPwZ7lLEHvi3cy6eMRyPlUAZPkSiPzuMD8jwEjaJ7zKB0hXeKkEYhyLNDZGY0VdLQcdt0nRAWUPvwacaN3EzU9ZrIs+Pghtt4bAGJaVN138w/Od4ZowdSoi6evYq+BQ8hyg00twE4/Qigu9YvrTMMX9Gzu7ORyEnRL5xjbeHWxai0SCWouJik/ifhzmDc6EePBsEU4Aldh9dNQxuFNIW0zBDpIXVg2mTGQYcC9mKspyVOYP+NHfE7l864anxuwa3kKz12V9B8EJrS2m5ULClfArerWcuCm0h4xW1Jf2Rei26LXC01Wj3B8jdgX7TFZxYuyig1JdRNlNaCZ9QKGtKsq1ZmCi3L6WgbByrYG95SLa4AKKoYPTRINgWdBZMcGAWM7kn+RMtEbfynLNmTRLenCvtKUdg5suep5hRSUcVJ6z2vY9MpZOJSYQAdqpqMQzeAoodgQmnnqHJE4AQq0qXIHWceTi6aWUKVl95+7mZU0HE4/u6oK5W6uD8L3QPHTCnnHf0Drl+j3nCKERJo8UO4uwYFMZAU+lu8GbAYnr1IwS8MK962FaWE8gi1Sjqv7kqzrVaNNnQVR0NSaeAIPYE8WUoFWbAyxU2nBqKEK2CsrssS57AGOQ2BWOb3+XwQZ2aQEFm3UDq2PjErHfdCEylf2FgLQ8vNnOapGe2fTL+Y9egMVMS4OEkTbKN+vGV90BLWMglsO84xdu4l44e7siWnWSWXqZ+SQ4oDJ6CWwvUzfpm9BfxHyuVVvv9hYXqO2/GF78f0NCe1/9Gt4NtfweToICnIeV+62XVHdluTj3FEhXKVZ7EZGsBjeV4FgYbioEJ3OlXVY026hrAGXgZbmsvq6XPcxpNso669/2naZ+ad1Ijt7tvZIUcSDO2iliwmgIOos0ZARYGpYDAApeqsrAU3lddVqw5aYsret05h/88CPdQAu+tDt5Trh5lHccy9zXJSjbtlSVfHQx0s6E4xeSjTLqFACgi72m9+xrMsoLJAWFhVLyG0+/JPl8GYCLivMeL41FCXn30e7ktrg9jZ3LVECdrNo0t0tDhikLLUOsf4WD0WP3MZjtTGzIzEDWbPwgorJ7dSeCH4qIYlbajpMBMQadeMUMU3GjZ+rs0loLjN3AEqmvLH2I2Y2y8pviVpSz4r2xPcLtzOTVzHhtuvWn+4o7ZzzcVHMQ3JoUs/g64xf6OuPgYE5JPdt8Yff+wHTphyyTkDsZkSo227ZmTVD+/MrCDqx32XDJT7EeMRHX+53sQXoZ9hjr1e4kPrlJdBZEVimTkaGvNmIEClGJZIfOAtgpD0MdWuKoZC9FOoA+xY821KmQkCXitmrfhxXGK0yP8uW0iWXHOtvQhOZ3G9r5lD9aqN25FzvMyMql/AZFJVOH36Z8eUiZ+XaWNjRZg6yXYe1u9WRrHH3w+Yv/LeNpi7rCr5jm5/wBgoAL3hLSwhsEXCL8ZgsOhx9ZXixhloB6tqwv4DgRF24llaiaPsOkMEL2z7qoBDuigds7ePKLmMByo9hpcI7aJpshrjfCoUuKqc+ty4GCcVkiI1ka91osHaKYPXczpaBniCU3fNgBST39xVdCZWbKhgd5+6GmbkhGJ6Rxa16xflXLNI4cvaomhO5Jz4cf7+FVMvwqt+yhSMV3cDndBWFot4a1pyuutNx2hVM3ZpDrdKIYzAlMq6c8fe8r9Y3dpdT8jrMiiLrN2h9kpf94vUhhgUz+mqCxUqWi9lrN8JnNm70KhjqSFnOoP/I7SvFIO9af+sGuF/pl/kv37IFtsmcbT6hyz8ljtpxazoW4QR7UcuDxLE9ftL8kRFSHFiiEiSBybhHn+Y695XKhoQVA3SeOn8PkqjJZwwdxTRYjoBeUEytuMnWAMIHJo4OGqcbNVoIZS2yl/Pbm+a25mmoSlD4WCOdGsbTsvG4sUEPUiGvQ2/jBzfzl4vZmsZgfuB0MKrlx1vo3T9kxzVCSAU4JQtxjyrT0DhKHLQ0qiF4Fnx2GGt5YYmqCWys7pq1WneCCqdUNtHJr31aQYe+lZnM2fn+AxQbgycglmvcB+ws7EJc5RhizCuO9F6UNxLq5W4swZb5qedGC8wmzpABiYXaUCCiCCcrRXltYm3LSKYwWzLwaG/qgTyJUrLuKxx3dUJNSPPyilz5D0Ntsgv5+A35PWjm4HnTrna0E8eI22OWu3eB/lCCTyS43McZ869N++wsGz6qK+l25Gbu6CksUUQRp6VglOf7CQzZKlymTW6Zo3P7ChAKAhD7PVlVcCA2dngpc2x2xehzxoUw0A+XuGzSx0PBivMryPkDDzoELkK3Bzo4CpU72anf3onu4cQ+8iht3Mdh5h5CGYG/3SVFN696+I4Gd60On42drkfD2SX77aNbJQG7nSyMX/I7XUydyXZw9VK/snK0nUhe2aAKrlIiTLdlGCMbqYXwlwlTjMaHb2I2qDKL81ge//qe7GfEJp/oHAaJ3+EVok2YSo9FLvALSBOo68oJa/Jx3WVs0lGexajoREirjmSzwVLciYoAozXOUhjj73nxOidA5fl8d4k1/CtA3HhXy7uUgjHsHC1PWu5eHiJjmedFO8DCVvnt5oBbm8mPsCS76u++9TMjZzsnZjlmYQuL9gojEfcjoBpqypSNhqNh4LxfK688pyb+PkylI3stNXQDM6UJAMiR7HrN0ipb3ayt9nKsixD0ymjzwXm50x7CPDxY/O9/VSUa2jUmzW9Ndn1t7pf5RhLu0K7IjqrvxyQvEquXs3en7kIljMPae+GmX4SEZdOzmiSClctDulj3xxa8F/AI77tIVPEhLk2UFrlQCbyMc9/6jin/vf8qmBUPm/mM9rhew/8Gff7Dza+pf/3F5/RGiqbw01WKKS0nhi0y+e/JeQ3+mMfHlwUQpNao20u9GpdQunXkZTlV3EI15DidqHgXN1G2jLJag9uFlnVV1Qe1T13Vu+QeR7biws1LqQpqJvDpVH1DrqLD6O7X+nBZE//s531JgP9Cm2DzJ131TRmhOumiZOf2aC6ydBRB76xP2VtTvKUr9ko5Nvi11b6vUxPCQMdO1C6bEDrGUELh0kSiZ+tr6KXM/qTl880WkLc3gMnKq4oDnUzPWRQZaDq2kC3QrrNusIdgS34SsVc5c9r4ZAfpb11vO5/OBT7Ozm85GFpcGztXCpUzeMGDjuLqldyjjhIUOdtc336vg4fvbRfRq+XnQ5S2i9fnpgdNFQfjpLqqmR/fCLhaI3jfQhcL9C2sgHQzs63lJkH/hiijiANNtNiuw9ln8y0CUq0oSPDmTRKY9xTE6+39zWzGTnk4AAA==")))
    print("SUCCESS: src/classification/train_distilroberta.py ready.")

if not eval_py.exists():
    print("Unpacking eval_transformer.py...")
    eval_py.write_bytes(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/51YW2/bOBZ+168gtCggY2RlOtiHnQBaIG3sIos0CWx3XoKAoCXK4UQStSSVNtvNf5/DmyT6kg3WKBqJPJfvXHmoOI4jKYqzoiZSsooVRDHentFnUmMlSCsrLhoqsu4lyt/1ixbA2hspiLQl+sybjgh4f6bokpFdy6ViBVq0O9ZSBNLREh7mm76lJdqMGtFXXtJaZlF0zUkpUaWplKG6ZCCiXvFPi9WGoEbTGVWKP9GW/Qd4K8EbVDL5lCJq4VCJAJB6pNFj35B2/kwFWAvCvvC6pC2aoF5TlRp5hYEOnCUTtFD1CyI7wlqp0N0jkRT9jrbwpwZgMkq+kj+5YOoFfXJrKdos51eXS/QLuoZXItD6j6/TRb5jxhUrugMlElTPsiiGeEQGPsZVr3pBMUas6bhQgKnlymCUUeTXxA4wSurf/5S89c8134GTd1ZcR9RjzbZe1h282g310gGRX79oX1JwcKEGDR24gkgE/7rSQYOEycKEybY9q0u8M87EdHAmllShAQ45QXFSKmQga3Fp4i34lgpFvLQkQvC7XCwvvl1v8Jfb68vFDf68/iMN1m+/be6+bfDl1cqu+2yY5jYGEA4WYLF0O6pwTba0xg3ptH9kGs2iSHsU0iv3rs2A7tqsJRi3pIFgAZVX/ulivbi+ulms8b/WtzfAFZdEkbNO8ALiTcszEF1TLPtOWxS+YdYq2qqpoyBJ+lrJTIc4HpRsVhc36+Xt6utihVeLNSz9X+oCH5/WGkUlrWwkdUFiU5DYlGBi/gdB4hxJJdB/TY4BjMNYzND8n0j1oP/e5NuQdPfAaF4fHs5NHKAcdPlPq38SOfSdst2jkulY+rZyG6qINn5sBJkuLC2xwZ2FpdGNmGdmk1UISszRZPQHOEUmMwtF/yAdoe6XrKY3XC1535YLIbhIqng5ArQNyfYMLl6MxErTIqLO0U8r/DWGRNEilXgZ5bvcBrbicVi0VTpaLYda7RU3XXLJxZr+u6dtQT8HBZQamo13ji2BHwXtFLoyMgx8Xduwum/mqm8Va6i1ML572WhUttFOwTQ9NMQtRboxkrrWEeImQ3S3DaJlHJPFM2sRaLQeKOkzKyhExJid2dckLvqSxDoidlm/Zkxi8kwYVGZNkxmC84GiuOj62IbPlmfG2oonJm90YwtPhXP0Qcapi7DlGgny0F2ZZsCdoKYN0TKB7Ewcp4veUY02Abw2OHfAIdaqQHvqFi0Ky5S/I6hvwsoUT7xYKxdKAWq11g52uX+GYpv3vkxccXtyUzk5+vnqa2KQcaQkvjOQyDvaJgNVimIBVgJyrv2Rx72q5v+AuEOeVSPnnjqNIdN5k1QOOljYi9Z6Jihxz+XakTunsYaCO300/46HkzlxVTa2e9fRzvc6jm38Ax/WeIy/TrSzsLvD8aCbWihy6GFfaEthBqIOqp2GOio0IgIBRg1Rgv1ApBBcwllb19ZqObStbdC2jqCc7cE/COL2VFMbI7h1efnu8B2omwZRH46JzTSd9D9fXVitDxhQ7odA4x3ke05oma/puApe82cjbCsi8bPEtZulILZ+lprwvVq9f0MXZTkZ2Uxvhe73ROH0MfhZu2dRxhRtAmeN4O89wId7kBBiN0j1PAAojJzRFWZMAHcAzywNGSCtBStkwMNhUIV0wH7P+HHkC0wL5uIx7T2v7rCHZfCGmuikwXEwLsR7gXOGx+GcnkyG+1k8iehodgB3L3b2HmGqZm7jj5Yfh4H8yDCNki3VJ5MLqDshhM/UvUAbLxxLo9ETwDsmn54BxYR1kpaHPrR2BcwnI3FSkDsrdcpP62Z8qqDu1MPDWPQ6ua0o6J5YFlzfZSDH9+EcZjm0CybNgQ7NKfG8UCWgbGZGgLj6ODersZHoKMLuoHD1UZvqNu9HnoeADhzpwOWhk41HvAXemRMuI37gtv4bdKTo1+zX2Z4pA7G3xYwQQBiItl6+t5oPK9tkbTgwVx8hfYUe8hIThkTbPkvR3/dq3LAeSbNDAca8UxIMwONK0RydYn2NDir63d1Ue8FSTI/mUZQ7is1dFMOlFNJJn4f+eprdQE+QHYERyPCbRT1wDQQXYtc3gOPO7MAMIwvBOj3v5LG7nNPTHwF8H3C3ebjCZ24mtJoyUpYallGRxPO5aWVzGNJjPYZVBAowP3JReUuEvTPOC/l8RMZ4IX1TxtCF5mYEO5QTThlvyuK9mrtWckTQqXvibBpPJ3kaRRfYBvxrQ3rDWxdEfwkGI1jxmbcV2yU1faZ17neubpa3KTJTjsrjDwmRhb5SzCS6/2BJ9Vkxkw/oQ2KfYEKGQVJKsoMXF0INBFIlRDXMzAeTobmeH7ugatZs78bnLv5l5bmOfqCwrG4LAm55cYpY+Zv5TADch58MPEzzMc1FBgj/9zeIZChbgza3dg6Lg735aPmwORiUD0/jpoeb+we7dTCYAcq3h+qpSakJUBbOpE4k5GQwuBpKveh4ZwGVTjzd9psniE9iX2S+ET1cKczcivmTebVs4+DqBcDo+v0do6uZUsu+6ZKf8RhoaKahWfHoENgbX14hpSH0LXhX5b8duXquyTN0KBp+CJ14V3F3EfS4dZXBQeW/IaE8RzHGuuYwji1uW4DRX0g42newFQAA")))
    print("SUCCESS: src/classification/eval_transformer.py ready.")

# 3. Resolve uploaded dataset files
TRAIN_CSV = TARGET_DATA_DIR / "apple_support_intent_training_candidates.csv"
GOLDEN_CSV = TARGET_DATA_DIR / "apple_support_intent_golden_set.csv"
BASELINES_JSON = TARGET_DATA_DIR / "apple_support_intent_evaluation_results.json"

search_roots = [Path("/content"), Path.cwd(), Path.cwd().parent, Path("/content/drive/MyDrive")]

# Scan for any uploaded apple_support folder anywhere in /content
print("Scanning for uploaded AppleSupport datasets in /content...")
found_folders = []
for root in search_roots:
    if root.exists():
        for p in root.glob("**/*"):
            if p.is_dir() and "apple" in p.name.lower() and "support" in p.name.lower():
                if p.resolve() != TARGET_DATA_DIR.resolve():
                    found_folders.append(p)

for folder in set(found_folders):
    print(f"Found uploaded dataset folder at: {folder}")
    for item in folder.iterdir():
        if item.is_file():
            dest = TARGET_DATA_DIR / item.name
            if not dest.exists() or dest.stat().st_size != item.stat().st_size:
                shutil.copy(item, dest)
                print(f"  Copied {item.name} ({item.stat().st_size / (1024*1024):.2f} MB) -> {dest}")

# Also search directly by file names across search roots if needed
if not TRAIN_CSV.exists():
    for root in search_roots:
        if root.exists():
            matches = [p for p in root.rglob("*apple_support_intent_training_candidates.csv") if p.resolve() != TRAIN_CSV.resolve()]
            if matches:
                shutil.copy(matches[0], TRAIN_CSV)
                print(f"Copied training candidates from {matches[0]}")
                break

if not GOLDEN_CSV.exists():
    for root in search_roots:
        if root.exists():
            matches = [p for p in root.rglob("*apple_support_intent_golden_set.csv") if p.resolve() != GOLDEN_CSV.resolve()]
            if matches:
                shutil.copy(matches[0], GOLDEN_CSV)
                print(f"Copied golden set from {matches[0]}")
                break

if not BASELINES_JSON.exists():
    for root in search_roots:
        if root.exists():
            matches = [p for p in root.rglob("*apple_support_intent_evaluation_results.json") if p.resolve() != BASELINES_JSON.resolve()]
            if matches:
                shutil.copy(matches[0], BASELINES_JSON)
                print(f"Copied baselines json from {matches[0]}")
                break
    if not BASELINES_JSON.exists():
        BASELINES_JSON.write_bytes(gzip.decompress(base64.b64decode("H4sIAAAAAAAC/+1cW4/jthV+31/BzkPRIp4J75Q2KJobgiyQBEUT9KUNBI5Nj9WVRFUXO06w/72HvowtjSnbM5Z3vWvPxrFNivx0eG7k+aA/XiF0k8e5SeLMRFNTlLHNbl6jG3KHbwausdK/2cym8+1Gum4sdDayaVQaM4LfOV38aKY6qXUFfaMqTk1Z6TRfXIWpvMXhLRG/YPmakNeE3XHFCMefYfwar4ZM7cgkJfT/A7657/q/toireXSvywXKx6Z15yjTqfv15sdVV/RNossSfb2+YLDufgwygaUK2Ray9ghlnsSVu/rBJiMDI8J4m46VrXQSrZoKM7TFyN0UEcFjl2FiSzOKZrZIRtFqYPi+3Vk8drYgfp0kUWqqIh6WW0KARj0c1oUezuFXfIdDzAabtlQPCxvlMGq8Wj7oggP6pAv0gAnWQ4RP2sdk2UYE2Wqbmfhh4mB3zvDYqzkJ29VlM8+q8d2jEHJTRHFWmazyysHW0Aj/N2WzzV3dRDjYbtvG1WgYk9sS1sPsaCrrPLeFUwFxhx8b3m3dk87zyBZgHsU0HpooLsvanA8VxR5Y97qqTDGPcjszxRnxSB+eOAE7fYhyPU9hbc+HCMx8N6KhTfNEg6pFY3Bt93r49oygiBdUlplhFU+dN8xMBY7jjLCUB9XILHR7oovRTBdnVO/AA2hsdFUXgMjOosqecd185gbeHPxWOYnzHLT8fIBCD57Sjiu3VFGdjyDinA8Qox5EdfY2s7MsstVkr0OCqEGpCLDkgkhCw934SBc+IoUMMKQeTFEVEBz4VpT7/Ho73noQLSMn0tN9qw7RkhDCmQwFJlxK5pV62PwjHatAIIUhASAiQmHFFPHdpvDFr3VMPugOILUhMhCKc0yk5B130LGArTvgoeQyUCFRgkPeERx2B0/yBvCc49phjVINacNvzawh0ffLjPPf26O3conGzDvjeqNHM8Q2m1rRrtG4I/C02nfEgEaPtj9uirfpGxttLTfVFHXLZTQam9b72PJrww5WYt8W8fZnhBr2+sF8FVvffh1cGHiKLxq9vGT0hF00enLJ6NUlgw8uWm8u2uOElwye0YtWHL6N/lX70yaVKxOX3SRmanxnYel9/FDbuozSOqni1XlRO3tdHM+5LUkjk8nsYjt0883iZO52cTKHNkd+6KfPv/oCPZ7UodLkuoBPyRxNY41clhmPTDY0n5sS8t3lNTrTybyMy7ub3VuJCu7hvl6cKE7LqJyneWVTH1zKW9lgAZn1IgMetHLXrf0JJ2r3sQvktyMNSeooLpdSWyD2za08U+OOqXdv0GEr4DLbxUnnFLJk34ytfP8FUy5UwWXd9WLgnbOJU01WGNgblaB0IM6p8YoTe+ZT/vnI7vnqxERO+cA0qrNoTjmBfVeU2bicH6stwj+dgg1wxyZuY9hDXZoyWhtWY1fRxDIzporiRWmCYEGaEq/Mb4vj+y+/yvPE/LzcTDojRolGWQ1mjvSwAoWJf9fD+D81xmOWDRB4kweL8sLeJybVJXIuwF0Ft6eL1CwGMAn6OqlNZWFLNHDD5Rq0IYEPpUFVoe81mqPMIuhexNnIuC6wYzSFm8WM9F1zgwXqNIKlNaONM2ttutp7xJUf2nEQsmmEvXDxEGc7+lQ2j1Zzgj9o74ybQl5ccAisndAa7e8Gh87StR19Og9+9jwdW+IDpvHF45tJnWpXFJrGZgbju6jjhHzzxhmJzYZxokG50MJp3S7RIHeQ5Gpaa7VC2ycBA1QOC2MyNIb3380Ags4IrY4g0KjQMbTYJLEzuAtoQ0uxoVlcTWwNY6MSGmDGkU3jDFQVrcLQ7pjls7FQCuWxse/NHDXs7A59r0coBXSL4wrEUGqzalIukGv01ixaTGJzd/qExhpkMUATuCaukAsecKs2RWVlC7AoO9LzxZWgdWBUw4nO4jJF92YSr350A8Yl2NwMlZnOYdQ79JNdz/43lDjh2jGKs6FNzRdwDfSGf4DQZhDfC/O/OgaTgLlscoe+hYgJ9+HWZoT+jGZOZpVFJUh0CBY/BBm/QQ+mQuM6SeDiMUTav6MfNAg1h94lcoauGRtQgf90NfarsTeN/d44s8pQPoEAB5kkWp/moZFO9QMEGV1DgopGdeEsGhIOHRcr09nSrG3tQ5Dk5XVljjNpgqVUgu+06R/nYJZTV3A3LghWzny+JEQEIgAQFJMA7M3MUPyPCUTHtc8Bm3L2uLhgBmpjnImtXVWiy6pEFCNwQ4C1fLRpJ5ciA3vKwCkUyJ16lsh9vNrO1XaatuOCGgJvDVZTOg1xfJC1PQ11BRpW2HwSD1vx8S/bulf+dal5Lpfd6N5Yx0kNCpvbsrpdCnTbml5tb1lXortJ7ENcVvEQbuIBNgVlc3/V4q/88t3tm2+/Q5+hH1ZXoX9urnoeiYVLFaoLJLGIkIk9JBappOgisUhJqY/EIhkJ95NYZLBdxtpJYmnhbJNYRCh5rywWScUzCqBKhpThx3fWM62F+KvIzA+SSyJYsHmXPdJcgoAyQUNOiJJcYemDK1qS86OXoZKKcxJIzKUgtFdOTOjD20RLO6StAipxIEMhBSWK902Y8Ra4qWr++RHTALYbfPPeP52mNSHx3UNnv8Y9dPY8MfdGtl4+9Ep0aAkW26YieqTm8CZa5YXrR+t1GSeh6gRKeDE1X0GHAjT8jgj6pPIs6RYb9oXXyXVIFKuQyc17j0SfplzAO73c2sSLeD4u1POABQqHUilOj6f5uIyFSYaZVAQDVq/XlpQETCkRwC6KhR1e2+UxjArKZMAJ57xHho9LhiCckRCgcR74Q6RXTE+WA5Ij6rpjxoVSV27PR8vtER9kzfHAWjVpXiWPKmse00q767stGKeo9hJxDL6jvrYpGacQ/Z7q/ekWZo8ceii0k87FflFlvQe9OeqrPB34984ukS/Roh50/kUsqfet8yfUanUKye+hI3a28qOoOt2ilme3WNLbwhB65fW8iNfD/HwJ8fERe7rIKELIk3N7OuajfXB7RMdyKti79sHvCTumDK8Enw+V4LN/N9gqqsHWnO6tYLoj4YCcqoi5F6MP5zPLjMcUTfEdoQ0ae6910wZ/9cr9uXJ/Tsb9OeDUaFsTBaPqAC/ARHhCL9CJ0IfymabZdcS20wsoJp851d6jwJ2uQHJ5pQZdBDWo4yD3ib5ycoBVEcbYqazKD84H8EyxjmMcnNWgCA8/Hb5QnBkNEpqmh9CEFp3Rz//68ZnsoIBiJi+RHcQDvo8dJDu5QZRwHzdIhPKAB9xIwfc94KaF8gk3SDzGwH64QaL5ks8hCjUqq5z1/vyboPXyVRl5R1lREEU5YSpkxFUW+6QNscbL+8yDbRqTe+9CTyUjQgUhV66o2ydvqIOWJVqIOzSkb6YQ9ZbJSdD86yg1U4Ibgj0DWcjPDjmCHtQnH6iDvtLFAPqEOT8dBtNJrjpoTU9C8eGMNv7JZ3B8uGRKBpCthxyShl4pPq3Xyw2Gv+xRPtvEFbKDgXwAS0ZiTEJClu9ebYH0g2MZYAEuKQhFJ0cGC1gQzGDMQPTJ73FP2MEhFYQEYHpe7H4ZPQ3EVEJPFoQyCMHtXgk+V4LPBRB8gv6eaNM/waddfsXnIviQ59Xc6ekYCy+padOzsx3o6Srw7H0/surK8LkyfPpm+HTrGP2wGT64m+GjrgyfT5jh00Gz4UReH91zSfSejgkVDWQf9J6gi1F0Jfd8JOQeLuQh5B5G8Psl9zic5yH38GdP1HE4sLP+qGh4JfdcyT0fALmHSn6AE8BhEL5Hbg+V/FzUHinIWZkIgRBXas9FUHs6zzxby8ooZoeYFWXyVGZ1LOfGQXyuUR3FI6KhVOdly1H58ZN7Xrn/3v0fW1zRR2xrAAA=")))
        print("Unpacked Phase 9 baseline evaluation results.")

# Verify both required datasets are in place
if not TRAIN_CSV.exists() or not GOLDEN_CSV.exists():
    print("DEBUG: Files currently found in /content:")
    for p in Path("/content").glob("**/*"):
        if p.is_file():
            print(f"  {p} ({p.stat().st_size / 1024:.1f} KB)")
    print("-------------------------------------------------")

assert TRAIN_CSV.exists(), f"Training candidates CSV missing at {TRAIN_CSV}"
assert GOLDEN_CSV.exists(), f"Golden evaluation set CSV missing at {GOLDEN_CSV}"

# 4. Load datasets and verify strict mathematical isolation
import pandas as pd
from src.classification.build_golden_evaluation_set import verify_training_isolation

df_train = pd.read_csv(TRAIN_CSV)
df_golden = pd.read_csv(GOLDEN_CSV)

verify_training_isolation(df_train, df_golden)
print(f"SUCCESS: Loaded {len(df_train)} training candidates and {len(df_golden)} golden evaluation records.")
print(f"Training dataset size: {TRAIN_CSV.stat().st_size / (1024*1024):.2f} MB")
print(f"Golden dataset size: {GOLDEN_CSV.stat().st_size / 1024:.2f} KB")
print("Strict training isolation verified: S_train ∩ S_golden = ∅")

In [ ]:
# Cell 4: Fine-Tune DistilRoBERTa on GPU
from src.classification.train_distilroberta import train_distilroberta

OUTPUT_DIR = "models/distilroberta_intent_classifier"

print("Starting GPU Fine-Tuning Pipeline for DistilRoBERTa on Tesla T4...")
model, tokenizer, metadata = train_distilroberta(
    train_csv_path=TRAIN_CSV,
    golden_csv_path=GOLDEN_CSV,
    output_dir=OUTPUT_DIR,
    model_name="distilroberta-base",
    epochs=3,
    batch_size=32,
    learning_rate=3e-5,
    max_length=128,
    seed=42,
)

print("\nFine-Tuning Complete!")
print(f"Model artifacts saved to: {OUTPUT_DIR}")

In [ ]:
# Cell 5: Comprehensive Evaluation on Golden Set & Comparison with Baselines
import json
from src.classification.train_distilroberta import evaluate_transformer_on_golden_set, get_label_mappings
from src.classification.build_golden_evaluation_set import load_golden_evaluation_set
from src.classification.eval_transformer import compare_with_phase9_baselines

golden_df = load_golden_evaluation_set(GOLDEN_CSV)
_, id2label = get_label_mappings()

eval_results = evaluate_transformer_on_golden_set(
    model=model,
    tokenizer=tokenizer,
    golden_df=golden_df,
    id2label=id2label,
    max_length=128,
)

# Compare with Phase 9 baselines
BASELINES_JSON = "data/processed/apple_support/apple_support_intent_evaluation_results.json"
comparison = compare_with_phase9_baselines(eval_results, BASELINES_JSON)

# Save comprehensive results JSON
RESULTS_JSON = "data/processed/apple_support/apple_support_distilroberta_evaluation_results.json"
with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump({"evaluation": eval_results, "comparison": comparison}, f, indent=2)

print(f"Evaluation results saved to: {RESULTS_JSON}")

# Print Comparison Table
print("\n=== Comparative Evaluation on Golden Set (155 Closed-World Records) ===")
print(f"{'Model':32s} | {'Accuracy':<10s} | {'Macro-F1':<10s} | {'Weighted-F1':<12s}")
print("-" * 70)
for m_key, m_info in comparison["models"].items():
    m_name = m_info["name"]
    met = m_info["metrics"]
    acc = met.get('accuracy', 0.0)
    mf1 = met.get('macro_f1', 0.0)
    wf1 = met.get('weighted_f1', 0.0)
    print(f"{m_name:32s} | {acc:<10.4f} | {mf1:<10.4f} | {wf1:<12.4f}")

# Print Per-Intent Deltas
print("\n=== Per-Intent F1 Comparison vs. Logistic Regression (Phase 9 Best) ===")
print(f"{'Intent':25s} | {'DistilRoBERTa':<14s} | {'LogReg':<10s} | {'Delta F1':<10s}")
print("-" * 65)
for intent, d in sorted(comparison["per_intent_deltas_vs_logistic_regression"].items()):
    sign = "+" if d['delta_f1'] >= 0 else ""
    print(f"{intent:25s} | {d['distilroberta_f1']:<14.4f} | {d['logistic_regression_f1']:<10.4f} | {sign}{d['delta_f1']:<10.4f}")

# Print Slice Performance
print("\n=== DistilRoBERTa Performance Across 8 Difficulty Slices ===")
print(f"{'Difficulty Slice':26s} | {'Accuracy':<10s} | {'Correct/Total':<15s}")
print("-" * 55)
for s_name, s_info in sorted(eval_results["slice_level_metrics"].items()):
    if "accuracy" in s_info:
        print(f"{s_name:26s} | {s_info['accuracy']:<10.4f} | {s_info['correct']}/{s_info['total']}")
    else:
        print(f"{s_name:26s} | {'N/A':<10s} | -/{s_info['total']} (ambiguous)")

# Print Ambiguous Cases Diagnostic
print("\n=== Ambiguous Cases Diagnostic (3 needs_review records) ===")
for amb in eval_results["ambiguous_cases_analysis"]:
    top_s = ", ".join(f"{t['intent']} ({t['confidence']:.3f})" for t in amb['top_predictions'])
    print(f"Tweet {amb['tweet_id']}:")
    print(f"  Text:            {amb['text'][:85]}...")
    print(f"  Top Prediction:  {amb['predicted_intent']} (confidence: {amb['confidence']:.4f}, margin: {amb['confidence_margin']:.4f})")
    print(f"  Top Predictions: {top_s}")
    print(f"  Notes:           {amb['human_reviewer_notes']}")

In [ ]:
# Cell 6: Package & Export Model Artifacts
import shutil

archive_name = "distilroberta_intent_classifier_artifacts"
print(f"Creating archive: {archive_name}.zip...")
shutil.make_archive(archive_name, 'zip', OUTPUT_DIR)
print(f"SUCCESS: Archive created at {archive_name}.zip")

try:
    from google.colab import files
    print("Downloading artifacts to local machine...")
    files.download(f"{archive_name}.zip")
    files.download(RESULTS_JSON)
except Exception as e:
    print(f"Note: If running in VS Code Colab extension, files are already in workspace: {OUTPUT_DIR}")